In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "id": "b6c17dda",
   "metadata": {},
   "source": [
    "# Exploration — Books to Scrape\n",
    "\n",
    "Freeform first-look analysis of the data produced by `scraper.py`.\n",
    "\n",
    "This notebook reads the outputs in `../data/` (CSV + SQLite) and pokes around\n",
    "before the polished charts get generated by `analysis.py`. Nothing here is\n",
    "meant to be \"final\" — it's for sanity-checking the scrape and spotting\n",
    "patterns worth turning into proper charts.\n",
    "\n",
    "**Inputs expected:**\n",
    "- `../data/books_raw.csv`\n",
    "- `../data/books_clean.csv`\n",
    "- `../data/books.db`\n",
    "\n",
    "Run `python scraper.py` from the project root first if these don't exist yet."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "00b81583",
   "metadata": {},
   "outputs": [],
   "source": [
    "import pandas as pd\n",
    "import numpy as np\n",
    "import sqlite3\n",
    "import matplotlib.pyplot as plt\n",
    "import seaborn as sns\n",
    "from pathlib import Path\n",
    "\n",
    "# Consistent look with analysis.py's charts\n",
    "sns.set_theme(style=\"whitegrid\")\n",
    "plt.rcParams[\"figure.figsize\"] = (9, 5)\n",
    "\n",
    "DATA_DIR = Path(\"../data\")\n",
    "RAW_CSV = DATA_DIR / \"books_raw.csv\"\n",
    "CLEAN_CSV = DATA_DIR / \"books_clean.csv\"\n",
    "DB_PATH = DATA_DIR / \"books.db\"\n",
    "\n",
    "for p in [RAW_CSV, CLEAN_CSV, DB_PATH]:\n",
    "    print(f\"{p}: {'found' if p.exists() else 'MISSING — run scraper.py first'}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "700ea52c",
   "metadata": {},
   "source": [
    "## 1. Load raw data\n",
    "\n",
    "Check what actually came out of the scraper before any cleaning."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "a0ada60b",
   "metadata": {},
   "outputs": [],
   "source": [
    "df_raw = pd.read_csv(RAW_CSV)\n",
    "print(df_raw.shape)\n",
    "df_raw.head()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "d5a8b004",
   "metadata": {},
   "outputs": [],
   "source": [
    "df_raw.info()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "4155dd1e",
   "metadata": {},
   "outputs": [],
   "source": [
    "# Quick data-quality pass: nulls, dupes, obviously-wrong types\n",
    "print(\"Nulls per column:\")\n",
    "print(df_raw.isna().sum())\n",
    "print()\n",
    "print(\"Duplicate rows:\", df_raw.duplicated().sum())\n",
    "if \"title\" in df_raw.columns:\n",
    "    print(\"Duplicate titles:\", df_raw['title'].duplicated().sum())"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "16332658",
   "metadata": {},
   "source": [
    "## 2. Load cleaned data\n",
    "\n",
    "This is what `utils.py`'s cleaning helpers produced — compare against raw to make sure nothing got mangled."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "0412e48f",
   "metadata": {},
   "outputs": [],
   "source": [
    "df = pd.read_csv(CLEAN_CSV)\n",
    "print(df.shape)\n",
    "df.head()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "4c1acefa",
   "metadata": {},
   "outputs": [],
   "source": [
    "df.dtypes"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "d1a1027c",
   "metadata": {},
   "outputs": [],
   "source": [
    "print(f\"Raw rows:   {len(df_raw)}\")\n",
    "print(f\"Clean rows: {len(df)}\")\n",
    "print(f\"Dropped:    {len(df_raw) - len(df)}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "7f395ca4",
   "metadata": {},
   "source": [
    "## 3. Price exploration"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "7599b9ff",
   "metadata": {},
   "outputs": [],
   "source": [
    "df['price'].describe()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "b62f8f38",
   "metadata": {},
   "outputs": [],
   "source": [
    "fig, ax = plt.subplots()\n",
    "sns.histplot(df['price'], bins=30, kde=True, ax=ax)\n",
    "ax.set_title(\"Price distribution\")\n",
    "ax.set_xlabel(\"Price (£)\")\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "3b519191",
   "metadata": {},
   "outputs": [],
   "source": [
    "# Cheapest and most expensive books — sanity check for scraping errors\n",
    "print(\"Cheapest:\")\n",
    "print(df.nsmallest(5, 'price')[['title', 'price', 'category']])\n",
    "print()\n",
    "print(\"Most expensive:\")\n",
    "print(df.nlargest(5, 'price')[['title', 'price', 'category']])"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "22eb128f",
   "metadata": {},
   "source": [
    "## 4. Category exploration"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "6c5dd1cf",
   "metadata": {},
   "outputs": [],
   "source": [
    "cat_counts = df['category'].value_counts()\n",
    "cat_counts"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "d829a6eb",
   "metadata": {},
   "outputs": [],
   "source": [
    "fig, ax = plt.subplots(figsize=(9, 6))\n",
    "cat_counts.head(15).plot(kind='barh', ax=ax)\n",
    "ax.set_title(\"Book count by category (top 15)\")\n",
    "ax.set_xlabel(\"Number of books\")\n",
    "ax.invert_yaxis()\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "7ae706e9",
   "metadata": {},
   "outputs": [],
   "source": [
    "avg_price_by_cat = df.groupby('category')['price'].mean().sort_values(ascending=False)\n",
    "avg_price_by_cat.head(15)"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "00ccf6c5",
   "metadata": {},
   "source": [
    "## 5. Rating exploration"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "a56c564b",
   "metadata": {},
   "outputs": [],
   "source": [
    "df['rating'].value_counts().sort_index()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "168a6014",
   "metadata": {},
   "outputs": [],
   "source": [
    "fig, ax = plt.subplots()\n",
    "sns.countplot(data=df, x='rating', order=sorted(df['rating'].unique()), ax=ax)\n",
    "ax.set_title(\"Rating distribution\")\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "baa6c104",
   "metadata": {},
   "source": [
    "## 6. Price vs. rating\n",
    "\n",
    "Is there any relationship, or are they independent?"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "41d78eec",
   "metadata": {},
   "outputs": [],
   "source": [
    "fig, ax = plt.subplots()\n",
    "sns.boxplot(data=df, x='rating', y='price', ax=ax)\n",
    "ax.set_title(\"Price by rating\")\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "1dac6ec6",
   "metadata": {},
   "outputs": [],
   "source": [
    "df.groupby('rating')['price'].agg(['mean', 'median', 'std', 'count'])"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "15884935",
   "metadata": {},
   "source": [
    "## 7. Availability\n",
    "\n",
    "books.toscrape.com exposes an in-stock count — worth a quick look if the scraper captured it."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "6f9e9176",
   "metadata": {},
   "outputs": [],
   "source": [
    "if 'availability' in df.columns:\n",
    "    print(df['availability'].value_counts().head(10))\n",
    "else:\n",
    "    print(\"No 'availability' column — skip, or check the raw scrape for the field name used.\")"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "ea0b7058",
   "metadata": {},
   "source": [
    "## 8. Cross-check against SQLite\n",
    "\n",
    "`scraper.py` also writes to `books.db` — confirm the CSV and DB agree before `analysis.py` runs its SQL queries against it."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "77097acf",
   "metadata": {},
   "outputs": [],
   "source": [
    "conn = sqlite3.connect(DB_PATH)\n",
    "tables = pd.read_sql(\"SELECT name FROM sqlite_master WHERE type='table'\", conn)\n",
    "print(tables)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "34b10ca0",
   "metadata": {},
   "outputs": [],
   "source": [
    "# Adjust table name below if different\n",
    "TABLE_NAME = \"books\"\n",
    "df_db = pd.read_sql(f\"SELECT * FROM {TABLE_NAME}\", conn)\n",
    "print(df_db.shape)\n",
    "df_db.head()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "f24ac3b8",
   "metadata": {},
   "outputs": [],
   "source": [
    "print(\"CSV rows:\", len(df))\n",
    "print(\"DB rows: \", len(df_db))\n",
    "assert len(df) == len(df_db), \"Row count mismatch between CSV and DB — investigate before trusting analysis.py output\"\n",
    "print(\"Row counts match ✅\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "196f4a05",
   "metadata": {},
   "outputs": [],
   "source": [
    "conn.close()"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "c2786ba3",
   "metadata": {},
   "source": [
    "## 9. Findings scratchpad\n",
    "\n",
    "Jot down anything worth writing up in the README or feeding into `analysis.py`:\n",
    "\n",
    "- Total books scraped: _fill in_\n",
    "- Number of categories: _fill in_\n",
    "- Price range: _fill in_\n",
    "- Most expensive category (avg): _fill in_\n",
    "- Cheapest category (avg): _fill in_\n",
    "- Any data-quality issues found (nulls, dupes, weird values): _fill in_\n",
    "- Anything surprising about the rating distribution: _fill in_\n"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "73b9805d",
   "metadata": {},
   "outputs": [],
   "source": [
    "# Quick summary stats to copy into the notes above / the README\n",
    "print(f\"Total books: {len(df)}\")\n",
    "print(f\"Categories: {df['category'].nunique()}\")\n",
    "print(f\"Price range: £{df['price'].min():.2f} – £{df['price'].max():.2f}\")\n",
    "print(f\"Highest avg price category: {avg_price_by_cat.idxmax()} (£{avg_price_by_cat.max():.2f})\")\n",
    "print(f\"Lowest avg price category: {avg_price_by_cat.idxmin()} (£{avg_price_by_cat.min():.2f})\")"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.11.0"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}